In [2]:
import numpy as np
from rnn_utils import *
from public_tests import *

In [ ]:
def rn_cl_fd(xt,aprev,pars):

  wmxminp=pars["wmxminp"]   #weight matrix mult input
  wmxmhd=pars["wmxmhd"]     #weight matrix mult hidd state
  wmxr=pars["wmxr"]         #weight matrix relat hidd state to ouptut
  ba=pars["ba"]
  by=pars["by"]

  nxa=np.tanh(np.dot(wmxmhd,aprev)+np.dot(wmxminp,xt)+ba)
  prdy=softmax(np.dot(wmxr,nxa)+by)       #imorted fun from file

  csh=(nxa,aprev,xt,pars)
  return nxa,prdy,csh

In [ ]:
#test

In [ ]:
#from ast import AnnAssign
def rn_fd(x,a0,pars):
  cchs=[]

  nx,m,tx=x.shape
  ny,na=pars["wmxr"].shape

  a=np.zeros((na,m,tx))
  ypred=np.zeros((ny,m,tx))
  nxa=a0

  for i in range(tx):
    nxa,ytpd,cache=rn_cl_fd(x[:,:,i],nxa,pars)
    a[:,:,i]=nxa
    ypred[:,:,i]=ytpd
    cchs.append(cache)
  cchs=(cchs,x)

  return a, ypred,cchs



In [ ]:
#test

In [ ]:
def lstm_cl_fd(xt,aprev,cprev,pars):

  wf=pars["wf"]
  bf=pars["bf"]
  wi=pars["wi"]
  bi=pars["bi"]
  wc=pars["wc"]
  bc=pars["bc"]
  wo=pars["wo"]
  bo=pars["bo"]
  wy=pars["wy"]
  by=pars["by"]


  nx,m=xt.shape
  ny,na=wy.shape

  concat=np.concatenate([aprev,xt])

  ft=sigmoid(np.dot(wf,concat)+bf)   #imported fun
  it=sigmoid(np.dot(wi,concat)+bi)
  cct=np.tanh(np.dot(wc,concat)+bc)

  nxc=cprev*ft + cct*it
  ot=sigmoid(np.dot(wo,concat)+bo)
  nxa=ot*(np.tanh(nxc))

  ytpred=softmax(np.dot(wy,nxa)+by)

  cch=(nxa,nxc,aprev,cprev,ft,it,cct,ot,xt,pars)



  return nxa,nxc,ytpred,cch



In [ ]:
#test


In [ ]:
def lstm_fd(x,a0,pars):
  cshs=[]
  nx,m,tx=x.shape
  ny,na=pars['wy'].shape
  a=np.zeros(na,m,tx)
  c=np.zeros(na,m,tx)
  y=np.zeros(na,m,tx)

  nxa=a0
  nxc=np.zeros((na,m))

  for t in range(tx):

    xt=x[:,:,t]
    nxa,nxc,yt,cch=lstm_cl_fd(xt,nxa,nxc,pars)
    a[:,:,t]=nxa
    c[:,:,t]=nxc
    y[:,:,t]=yt

  cchs.append(cch)
  cchs=(cchs,x)

  return a,y,c,cchs



In [ ]:
#test

In [ ]:
#B prop

def rnn_cl_bd(nxda,cch):

  (nxa,aprev,xt,pars)=cch
  wmxminp=pars["wmxminp"]
  wmxmhd=pars["wmxmhd"]
  wmxr=pars["wmxr"]
  ba=pars["ba"]
  by=pars["by"]

  dtanh=nxda*(1-(nxa)**2)

  dxt=np.dot(wmxminp.T,dtanh)
  dwmxminp=np.dot(dtanh,xt.T)

  daperv=np.dot(wmxmhd.T,dtanh)
  dwmxmhd=np.dot(dtanh,aprev.T)

  dba=np.sum(dtanh,axis=1,keepdims=True)

  gradients={"dxt":dxt,"daprev":daperv,"dwmxminp":dwmxminp,"dwmxmhd":dwmxmhd,"dba":dba}

  return gradients



In [ ]:
#test

In [ ]:
def rnn_bd(da,cchs):
  (cchs,x)=cchs
  (a1,a0,x1,pars)=cchs[0]

  na,m,Tx=da.shape
  nx,m=x1.shape

  dx=np.zeros((nx,m,Tx))
  dwmxminp=np.zeros((na,nx))
  dwmxmhd=np.zeros((na,na))
  dba=np.zeros((na,1))
  da0=np.zeros((na,m))
  daprev=np.zeros((na,m))

  for i in reversed(range(Tx)):
    gradients=rnn_cl_bd(daprev,da[:,:,i],cchs[i])
    dxt,daprev,dwmxminpt,dwaat,dbat=gradients["dxt"],gradients["daprev"],gradients["dwmxminpt"],gradients["dwmxmhdt"],gradients["dba"]
    dx[:,:,i]=dxt
    dwmxminp+=dwmxminpt
    dwmxmhdt=dwmxmhdt
    dba+=dbat
  da0=daprev

  gradients={"dx":dx,"da0":da0,"dwmxminp":dwmxminp,"dwmxmhd":dwmxmhd,"dba":dba}



  return

In [ ]:
#test

In [ ]:
def lstm_cl_bd(nxda,nxdc,cch):
  (nxa,nxc,aprev,cprev,ft,it,cct,ot,xt,pars)=cch
  nx,m=xt.shape
  na,m=nxa.shape

  tanhnxc=np.tanh(nxc)
  dot=nxda*tanhnxc*ot*(1-ot)
  dcct=(nxdc+nxda*ot*(1-tanhnxc**2))*it*(1-cct**2)
  dit=(nxdc+nxda*ot*(1-tanhnxc**2))*cct*it*(1-it)
  dft=(nxdc+nxda*ot*(1-tanhnxc**2))*cprev*ft*(1-ft)

  concat=np.concatenate((aprev,xt),axis=0)
  dwf=np.dot(dft,concat.T)
  dwi=np.dot(dwi,concat.T)
  dwc=np.dot(dwc,concat.T)
  dwo=np.dot(dwo,concat.T)

  dbf=np.sum(dft,axis=1,keepdims=True)
  dbi=np.sum(dit,axis=1,keepdims=True)
  dbc=np.sum(dcct,axis=1,keepdims=True)
  dbo=np.sum(dot,axis=1,keepdims=True)


  wf=pars["wf"]
  wi=pars["wi"]
  wc=pars["wc"]
  wo=pars["wo"]

  dconcat=

  daprev=dconcat[:na,:]
  dxt=dconcat[na:, :]

  dcprev=(nxdc+nxda*ot*(1-tanhnxc**2))*ft

  gradients={}


  return

In [ ]:
#test

In [ ]:
def lstm_bd(da,cchs):
  (cchs,x)=cchs
  (a1,c1,a0,c0,f1,i1,cc1,o1,x1,pars)=cchs[0]

  na,m,tx=da.shape
  nx,m=x1.shape

  dx=np.zeros((nx,m,tx))
  da0=np.zeros((na,m))
  daprevt=np.zeros((na,m))
  dcprevt=np.zeros((na,m))
  dwf=np.zeros((na,na+nx))
  dwi=np.zeros((na,na+nx))
  dwc=np.zeros((na,na+nx))
  dwo=np.zeros((na,na+nx))
  dbf=np.zeros((na,1))
  dbi=np.zeros((na,1))
  dbc=np.zeros((na,1))
  dbo=np.zeros((na,1))


  for i in reversed(range(tx)):
    gradients=lstm_cl_bd(daprevt+da[:,:,i],dcprevt,cchs[i])
    dx[:,:,i]=gradients["dxt"]
    daprevt=
    dcprevt=

    dwf+=gradients["dwf"]
    dwi+=gradients["dwi"]
    dwc+=gradients["dwc"]
    dwo+=gradients["dwo"]
    dbf+=gradients["dbf"]
    dbi+=gradients["dbi"]
    dbc+=gradients["dbc"]
    dbo+=gradients["dbo"]

  da0=daprevt

  gradients={"dx":dx,"da0":da0,"dwf":dwf,"dbf":dbf,"dwi":dwi,"dbi":dbi,"dwc":dwc,
             "dbc":dbc,"dwo":dwo,"dbo":dbo}
  return


In [ ]:
#test